# **Fine-Tune Pre-Trained Classification Model With QLORA**

### **Import Libraries**

In [27]:
import torch
import torch.nn as nn   
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig

In [3]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
device.type

'cuda'

In [ ]:
def save_data_to_file (data, file_path):
    
    with open(file_path, "w") as file:
        json.dump(data, file,indent=4)
    print(f"file created successfully at {file_path}")

In [6]:
def load_data_from_file(file_path):
    
    with open(file_path, "r") as file:
        load_file = json.load(file)
    
    return load_file

### **Load Dataset**

In [17]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [18]:
train_labels = dataset['train']['label']
num_classes = len(set(train_labels))
num_classes

2

In [19]:
imdb_labels = {0: 'negative review', 1: 'positive review'}

In [20]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

##### Tokenizer Map Function

In [21]:
def tokenized_text (data):
    return tokenizer(data['text'], 
                     padding='max_length', max_length=512, truncation=True)

In [22]:
dataset = dataset.map(tokenized_text, batched=True)

Map: 100%|██████████| 50000/50000 [00:30<00:00, 1617.79 examples/s]


In [23]:
dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [24]:
dataset = dataset.rename_column('label', 'labels')

In [25]:
dataset.set_format(type='torch', columns = ['labels', 'input_ids', 'attention_mask'])

In [26]:
dataset['train'][0]

{'labels': tensor(0),
 'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
          2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
          2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
          2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
          1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
          2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
          6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
          1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
          5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
         14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
          1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
          2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
         25430, 1

### **BitsAndBytes Configuration**

In [28]:
config_bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype= torch.bfloat16,
    llm_int8_skip_modules=['classifier', 'pre_classifier']
)